In [4]:


import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.callbacks import BaseCallback



In [2]:
SEED = 42
TOTAL_STEPS = 200_000           # bump to 300k–500k for stronger results
LOG_DIR = "runs_cartpole_dqn"
os.makedirs(LOG_DIR, exist_ok=True)
set_random_seed(SEED)

# --- Callback to capture episodic returns from Monitor wrapper ---
class CurveLogger(BaseCallback):
    def __init__(self):
        super().__init__()
        self.timesteps = []
        self.returns = []
    def _on_step(self) -> bool:
        info = self.locals.get("infos", [{}])[-1]
        if "episode" in info:
            self.timesteps.append(self.num_timesteps)
            self.returns.append(info["episode"]["r"])
        return True

# --- Make env with Monitor so we can read episode returns easily ---
def make_env(seed=SEED):
    env = gym.make("CartPole-v1")
    env = Monitor(env, filename=None)  # logs go to memory (callback reads from infos)
    env.reset(seed=seed)
    return env

env = make_env()

# --- Popular DQN config (double DQN + target network; dueling by default in SB3) ---
model = DQN(
    "MlpPolicy",
    env,
    learning_rate=2.5e-4,
    buffer_size=100_000,
    learning_starts=1_000,
    batch_size=64,
    gamma=0.99,
    train_freq=(4, "step"),
    target_update_interval=1_000,
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    policy_kwargs={"net_arch": [256, 256]},  # common MLP
    verbose=1,
    seed=SEED,
)

logger = CurveLogger()
model.learn(total_timesteps=TOTAL_STEPS, callback=logger)
env.close()

# --- Plot learning curve (smoothed) ---
def moving_avg(x, w=10):
    if len(x) < w:
        return np.array(x)
    return np.convolve(x, np.ones(w)/w, mode="valid")

t = np.array(logger.timesteps)
r = np.array(logger.returns)
r_s = moving_avg(r, w=10)
t_s = t[-len(r_s):]

plt.figure(figsize=(8,5))
plt.plot(t_s, r_s, label="DQN (smoothed)")
plt.xlabel("Timesteps")
plt.ylabel("Episodic Return")
plt.title("CartPole-v1 — DQN Learning Curve")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# --- Quick printout & optional save ---
print(f"Episodes: {len(r)}, Final episode return: {r[-1] if len(r)>0 else 'N/A'}")
np.savetxt(os.path.join(LOG_DIR, "returns.csv"), np.c_[t, r], delimiter=",", header="timesteps,return", comments="")


Using cpu device
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 23       |
|    ep_rew_mean      | 23       |
|    exploration_rate | 0.995    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 14105    |
|    time_elapsed     | 0        |
|    total_timesteps  | 92       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 24.9     |
|    ep_rew_mean      | 24.9     |
|    exploration_rate | 0.99     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 16452    |
|    time_elapsed     | 0        |
|    total_timesteps  | 199      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 26.8     |
|    ep_rew_mean      | 26.8     |
|    exploration_rate | 0.984    |
| t

KeyboardInterrupt: 